In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from data.dataset import MalwareDatasetLoader

loader = MalwareDatasetLoader()
df = loader.df
df_train, df_val, df_test = loader.make_data_splits()

/home/a75wu/STAT-841-cr-an-di/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using file: /home/a75wu/.cache/kagglehub/datasets/agungpambudi/network-malware-detection-connection-analysis/versions/3/CTU-IoT-Malware-Capture-1-1conn.log.labeled.csv
Using file: /home/a75wu/.cache/kagglehub/datasets/agungpambudi/network-malware-detection-connection-analysis/versions/3/CTU-IoT-Malware-Capture-20-1conn.log.labeled.csv
Using file: /home/a75wu/.cache/kagglehub/datasets/agungpambudi/network-malware-detection-connection-analysis/versions/3/CTU-IoT-Malware-Capture-21-1conn.log.labeled.csv
Using file: /home/a75wu/.cache/kagglehub/datasets/agungpambudi/network-malware-detection-connection-analysis/versions/3/CTU-IoT-Malware-Capture-3-1conn.log.labeled.csv
Using file: /home/a75wu/.cache/kagglehub/datasets/agungpambudi/network-malware-detection-connection-analysis/versions/3/CTU-IoT-Malware-Capture-34-1conn.log.labeled.csv
Using file: /home/a75wu/.cache/kagglehub/datasets/agungpambudi/network-malware-detection-connection-analysis/versions/3/CTU-IoT-Malware-Capture-35-1conn.log.

/fsys1/home/a75wu/STAT-841-cr-an-di/data/dataset.py:28: DtypeWarning: Columns (8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframes.append(pd.read_csv(full_path, sep="|"))


Using file: /home/a75wu/.cache/kagglehub/datasets/agungpambudi/network-malware-detection-connection-analysis/versions/3/CTU-IoT-Malware-Capture-60-1conn.log.labeled.csv
Using file: /home/a75wu/.cache/kagglehub/datasets/agungpambudi/network-malware-detection-connection-analysis/versions/3/CTU-IoT-Malware-Capture-8-1conn.log.labeled.csv
Using file: /home/a75wu/.cache/kagglehub/datasets/agungpambudi/network-malware-detection-connection-analysis/versions/3/CTU-IoT-Malware-Capture-9-1conn.log.labeled.csv


/fsys1/home/a75wu/STAT-841-cr-an-di/data/dataset.py:31: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace("-", np.nan)


Train: 17507702
Val: 3751650
Test: 3751651


### Misc

In [4]:
# which cols have NaNs
df.isna().any()

ts                False
uid               False
id.orig_h         False
id.orig_p         False
id.resp_h         False
id.resp_p         False
proto             False
service            True
duration           True
orig_bytes         True
resp_bytes         True
conn_state        False
local_orig         True
local_resp         True
missed_bytes      False
history            True
orig_pkts         False
orig_ip_bytes     False
resp_pkts         False
resp_ip_bytes     False
tunnel_parents     True
label             False
detailed-label     True
dtype: bool

In [5]:
# ONE_HOT_COLUMNS = ['proto', 'service', 'conn_state', 'history']
# # NUMERIC_COLUMNS = [
# #    'duration', 'orig_bytes', 'resp_bytes', 'missed_bytes', 'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes'
# # ]
# LABEL_COLUMN = ['label']

# for col in [*ONE_HOT_COLUMNS, *LABEL_COLUMN]:
#     uniques = df[col].unique()
#     print(f"\nColumn: {col} (nunique={len(uniques)})")
#     print("Sample uniques:", uniques)

In [6]:
cols_custom_na = df.columns[(df == "-").any(axis=0)]
print(list(cols_custom_na))

[]


# Preprocessing

In [8]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# local_orig, local_resp have only "-"" values
SKIPPED_COLUMNS = [
  'ts', 'uid', 'id.orig_h', 'id.resp_h', 'tunnel_parents', 'detailed-label', 'id.orig_p', 'id.resp_p', 'local_orig', 'local_resp']

ONE_HOT_COLUMNS = ['proto', 'service', 'conn_state', 'history']
NUMERIC_COLUMNS = [
   'duration', 'orig_bytes', 'resp_bytes', 'missed_bytes', 'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes'
]
LABEL_COLUMN = 'label'

num_transformer = Pipeline(
  [
    ("imputer", SimpleImputer(missing_values=np.nan, strategy="constant", fill_value=-1)),
    ("scalar", StandardScaler())
  ]
)
cat_transformer = Pipeline(
  [
    ("imputer", SimpleImputer(missing_values=np.nan, strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown='ignore', sparse_output=False))
  ]
)

preprocessor = ColumnTransformer(
    [("numeric", num_transformer, NUMERIC_COLUMNS),
    ("categorical", cat_transformer, ONE_HOT_COLUMNS)],
    remainder='passthrough'
)

def process_data(df, using_train_data):
    tmp_df = df[ONE_HOT_COLUMNS + NUMERIC_COLUMNS]
    # fit only on training data
    # only transforming for val and test data
    if using_train_data:
        X = preprocessor.fit_transform(tmp_df)
    else:
        X = preprocessor.transform(tmp_df)

    y = np.where(df[LABEL_COLUMN] == 'Benign', 1, 0)
    y = y.reshape(-1, 1)
    return X, y

In [9]:
X_train, y_train = process_data(df_train, using_train_data=True)
X_val, y_val = process_data(df_val, using_train_data=False)
X_test, y_test = process_data(df_test, using_train_data=False)

In [10]:
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(17507702, 259) (17507702, 1)
(3751650, 259) (3751650, 1)
(3751651, 259) (3751651, 1)


In [11]:
np.unique(y_train, return_counts=True)

(array([0, 1]), array([11361141,  6146561]))

In [21]:
from data.dataset import MalwareDataset
from torch.utils.data import DataLoader

train_ds = MalwareDataset(X_train, y_train)
val_ds = MalwareDataset(X_val, y_val)
test_ds = MalwareDataset(X_test, y_test)

BATCH_SIZE = 4096

train_dataloader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False)
val_dataloader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_dataloader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# Neural Network Experiments

In [26]:
import sys, os
import matplotlib.pyplot as plt
from tqdm import tqdm
import logging

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader


class Trainer:
    def __init__(
        self,
        model: nn.Module,
        optimizer: optim.Optimizer,
        batch_size: int = 256,
        learning_rate: float = 0.01,
        num_epochs: int = 30,
        check_val_every_n_epoch: int = 1,
        device: torch.device = torch.device("cpu"),
        threshold: float = 0.5,
    ) -> None:
        """Trainer object to facilitate training and evaluation"""

        self.model = model

        # training configurations
        self.batch_size = batch_size  # does nothing; mainly for viz
        self.learning_rate = learning_rate
        self.num_epochs = num_epochs
        self.check_val_every_n_epoch = check_val_every_n_epoch
        self.device = device
        self.threshold = threshold
        self.model.to(self.device)

        # set loss function and optimizer
        self.criterion = nn.BCEWithLogitsLoss()

        self.optimizer = optimizer
        self.optimizer_name = self.optimizer.__class__.__name__
            
        self.scheduler = optim.lr_scheduler.StepLR(self.optimizer, step_size=7, gamma=0.1)

        self.model_name = f"DNN (lr={self.learning_rate}, bs={self.batch_size}, loss={self.optimizer_name}, threshold={self.threshold})"

        # model metrics
        self.train_losses = []
        self.train_accuracies = []
        self.val_losses = []
        self.val_accuracies = []

        # logging info
        logging.basicConfig(stream=sys.stdout, level=logging.INFO, format="%(levelname)s | %(message)s")
        self.logger = logging.getLogger()

    def train(self, train_dataloader: DataLoader, val_dataloader: DataLoader) -> None:
        """Train the ResNet-18 Model"""

        for epoch in range(self.num_epochs):
            self.model.train()  # set model to train

            # loss tracking metrics
            running_loss = 0.0
            running_vloss = 0.0
            batch_loss = 0.0
            running_acc = 0.0

            pbar = tqdm(enumerate(train_dataloader), total=len(train_dataloader))

            for i, (inputs, labels) in pbar:
                inputs, labels = inputs.to(self.device), labels.to(self.device)

                # zero gradients for every batch
                self.optimizer.zero_grad()

                # compute predictions + loss
                outputs = self.model(inputs)  # predicted class
                loss = self.criterion(outputs, labels)

                # compute training accuracy
                running_acc += self.__accuracy(outputs, labels)

                # perform backpropagation
                loss.backward()  # compute gradients
                self.optimizer.step()  # update model parameters

                # gather data and report
                running_loss += loss.item()
                batch_loss += loss.item()
                if i % 10 == 0:
                    batch_loss = batch_loss / 10  # loss per batch
                    pbar.set_postfix({"loss": round(batch_loss, 5)})
                    batch_loss = 0.0

            self.scheduler.step()

            train_accuracy = running_acc / len(train_dataloader)
            self.train_accuracies.append((epoch, train_accuracy))

            avg_loss = running_loss / len(train_dataloader)
            self.train_losses.append((epoch, avg_loss))

            if epoch % self.check_val_every_n_epoch == 0:
                self.model.eval()  # set model to evaluation
                with torch.no_grad():
                    running_val_acc = 0
                    for inputs, labels in val_dataloader:
                        inputs, labels = inputs.to(self.device), labels.to(self.device)

                        outputs = self.model(inputs)
                        loss = self.criterion(outputs, labels)

                        running_vloss += loss.item()
                        # compute validtion accuracy
                        running_val_acc += self.__accuracy(outputs, labels)

                val_accuracy = running_val_acc / len(val_dataloader)
                self.val_accuracies.append((epoch, val_accuracy))

                avg_vloss = running_vloss / len(val_dataloader)
                self.val_losses.append((epoch, avg_vloss))

                self.logger.info(
                    f"[EPOCH {epoch + 1}] LOSS : train={avg_loss} val={avg_vloss} | ACCURACY : train={train_accuracy} val={val_accuracy}"
                )

    def test(self, test_dataloader: DataLoader) -> None:
        """Test the ResNet-18 Model"""

        correct = 0
        self.model.eval()
        with torch.no_grad():
            for inputs, labels in test_dataloader:
                inputs, labels = inputs.to(self.device), labels.to(self.device)
                outputs = self.model(inputs)
                correct += self.__accuracy(outputs, labels)

        self.logger.info(f"Test accuracy: {(correct / len(test_dataloader)) * 100} %")

    def plot_metrics(self) -> None:
        """Create plots for model metrics"""

        os.makedirs("plots", exist_ok=True)  # create plots dir

        t_iters, t_loss = list(zip(*self.train_losses))
        _, v_loss = list(zip(*self.val_losses))
        _, acc = list(zip(*self.train_accuracies))
        _, v_acc = list(zip(*self.val_accuracies))

        fig, ax = plt.subplots(1, 2, figsize=(12, 5))
        fig.suptitle(f"Model: [{self.model_name}]")

        ax[0].set_title(f"Loss Curve (batch_size={self.batch_size}, lr={self.learning_rate})")
        ax[0].plot(t_iters, t_loss)
        ax[0].plot(t_iters, v_loss)
        ax[0].set_xlabel("Epochs")
        ax[0].set_ylabel("Loss")
        ax[0].legend(["Train", "Validation"])
        ax[0].set_xticks(t_iters)

        ax[1].set_title(f"Accuracy Curve (batch_size={self.batch_size}, lr={self.learning_rate})")
        ax[1].plot(t_iters, acc)
        ax[1].plot(t_iters, v_acc)
        ax[1].set_xlabel("Epochs")
        ax[1].set_ylabel("Accuracy")
        ax[1].legend(["Train", "Validation"])
        ax[1].set_xticks(t_iters)

        fig.savefig(f"plots/{self.model_name}_metrics.png")
        plt.show()

    def __accuracy(self, outputs: torch.Tensor, labels: torch.Tensor) -> float:
        probs = torch.sigmoid(outputs)
        preds = (probs >= self.threshold).int()
        return (torch.sum(preds == labels) / len(preds)).item()

In [27]:
import torch
import torch.nn as nn

class DNN(nn.Module):
    def __init__(self, 
                input_dim: int, 
                hidden_dims: list[int] = [512],
                output_dim: int = 10,
                dropout: float = 0.0
        ):
        super().__init__()
        layers = []
        prev_dim = input_dim

        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())

            if dropout > 0.0:
                layers.append(nn.Dropout(dropout))

            prev_dim = h

        layers.append(nn.Linear(prev_dim, output_dim))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

def get_device():
    """Get available device"""

    if torch.cuda.is_available():
        print("Using CUDA...")
        return torch.device("cuda")
    elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
        print("Using MPS...")
        return torch.device("mps")
    else:
        print("Using CPU...")
        return torch.device("cpu")
    
device = get_device()

Using CUDA...


In [28]:
# 2 class output -> malware or benign
model_2l= DNN(input_dim=X_train.shape[1], hidden_dims=[512], output_dim=1)
print(model_2l)

DNN(
  (model): Sequential(
    (0): Linear(in_features=259, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=1, bias=True)
  )
)


### Stochastic Gradient Descent

In [ ]:
import torch.optim as optim

learning_rate=1e-2
optimizer = optim.SGD(model_2l.parameters(), lr=learning_rate) # optimizer

sgd_trainer_model_2l = Trainer(
    model_2l,
    optimizer,
    batch_size=BATCH_SIZE,
    learning_rate=learning_rate,
    num_epochs=4,
    check_val_every_n_epoch=3,
    device=device
)

sgd_trainer_model_2l.train(train_dataloader, val_dataloader)

  0%|          | 0/4275 [00:00<?, ?it/s, loss=0.0689]

 68%|██████▊   | 2928/4275 [01:06<00:32, 41.99it/s, loss=0.532]

In [ ]:
sgd_trainer_model_2l.test(test_dataloader)

In [ ]:
sgd_trainer_model_2l.plot_metrics()

In [ ]:
model_2l_dropout = DNN(input_dim=X_train.shape[1], hidden_dims=[512], output_dim=1, dropout=0.5)

learning_rate=1e-2
optimizer = optim.SGD(model_2l_dropout.parameters(), lr=learning_rate) # optimizer

sgd_trainer_model_2l_dropout = Trainer(
    model_2l_dropout,
    optimizer,
    batch_size=BATCH_SIZE,
    learning_rate=learning_rate,
    num_epochs=4,
    check_val_every_n_epoch=3,
    device=device
)
sgd_trainer_model_2l_dropout.train(train_dataloader, val_dataloader)

SimpleMLP(
  (model): Sequential(
    (0): Linear(in_features=86, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=512, out_features=1024, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.5, inplace=False)
    (6): Linear(in_features=1024, out_features=1, bias=True)
  )
)


In [ ]:
sgd_trainer_model_2l_dropout.test(test_dataloader)

In [ ]:
import torch.optim as optim

criterion = nn.BCEWithLogitsLoss() # Loss function
learning_rate = 1e-2
optimizer = optim.SGD(model.parameters(), lr=learning_rate) # optimizer
epochs = 6
check_val_every_n_epoch = 5

train(model, epochs, optimizer, criterion, train_dataloader, val_dataloader, check_val_every_n_epoch)

100%|██████████| 2344/2344 [00:07<00:00, 326.52it/s, loss=0.675]


[EPOCH 1] LOSS : train=0.6839488231332229 val=0.7005257393284 | ACCURACY : train=83.0285873413086 val=83.06082916259766


100%|██████████| 2344/2344 [00:06<00:00, 372.87it/s, loss=0.639]


[EPOCH 6] LOSS : train=0.640104876875674 val=0.7314943316913392 | ACCURACY : train=83.0285873413086 val=83.06082916259766


 59%|█████▉    | 1383/2344 [00:04<00:02, 344.95it/s, loss=0.613]


KeyboardInterrupt: 

In [ ]:
# with dropout
model = SimpleMLP(input_dim=X_train.shape[1], hidden_dims=[512, 1024], output_dim=1)
model = model.to(device)
print(model)

criterion = nn.BCEWithLogitsLoss() # Loss function
learning_rate = 1e-2
optimizer = optim.SGD(model.parameters(), lr=learning_rate) # optimizer
epochs = 6
check_val_every_n_epoch = 5

train(model, epochs, optimizer, criterion, train_dataloader, val_dataloader, check_val_every_n_epoch)

In [ ]:
# with dropout
model = SimpleMLP(input_dim=X_train.shape[1], hidden_dims=[512, 1024], output_dim=1, dropout=0.5)
model = model.to(device)
print(model)

criterion = nn.BCEWithLogitsLoss() # Loss function
learning_rate = 1e-2
optimizer = optim.SGD(model.parameters(), lr=learning_rate) # optimizer
epochs = 6
check_val_every_n_epoch = 5

train(model, epochs, optimizer, criterion, train_dataloader, val_dataloader, check_val_every_n_epoch)

### Adam Optimizer

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=learning_rate) # optimizer